In [1]:
from ugot import ugot

got = ugot.UGOT()
got.initialize("192.168.88.1")
got.show_light_rgb([1, 2, 3], 255, 255, 255)

192.168.88.1:50051


In [2]:
got.load_models(['color_recognition'])

True

In [ ]:
# Task 1: Rotate UGOT to find the blue cube
result = got.get_color_total_info()
print(result)

while result[0] != 'Blue':
    result = got.get_color_total_info()
    got.mecanum_turn_speed(3, 45)

got.mecanum_stop()

['', '', -1, -1, -1, -1, -1]


In [ ]:
# Before adding traffic light behaviour
import cv2
import numpy as np

got.open_camera()

try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get color recognition result: [color, shape, center_x, center_y, height, width, area]
        result = got.get_color_total_info() 

        if result:
            # Unpack into several variables
            color, shape, cx, cy, h, w, area = result
            cx, cy = int(cx), int(cy)
            half_w, half_h = int(w / 2), int(h / 2)

            # Compute bounding box corners
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # Draw bounding box and center dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label with color, shape, and area
            lines = [
                f"{color} {shape} width: {w}, area: {area}",
                f"cx: {cx} cy: {cy}",
            ]

            line_height = 25  # adjust based on font scale
            for i, line in enumerate(lines):
                y = (y1 - 10) - (len(lines) - 1 - i) * line_height
                cv2.putText(img, line, (x1, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Display the frame
        cv2.imshow("Colour Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    got.mecanum_stop()

    cv2.destroyAllWindows()
    cv2.waitKey(1)

In [ ]:
# After adding traffic light behaviour
import cv2
import numpy as np
from IPython.display import display, Image, clear_output

GREEN_SPEED = 30
YELLOW_SPEED = 10

got.open_camera()
try:
    while True:
        # Grab the latest camera frame as raw bytes
        frame = got.read_camera_data()

        # Decode the bytes into an OpenCV image array
        nparr = np.frombuffer(frame, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

        # Get color recognition result: [color, shape, center_x, center_y, height, width, area]
        result = got.get_color_total_info() 

        if result:
            # Unpack into several variables
            color, shape, cx, cy, h, w, area = result
            cx, cy = int(cx), int(cy)
            half_w, half_h = int(w / 2), int(h / 2)

            # Compute bounding box corners
            x1, y1 = cx - half_w, cy - half_h
            x2, y2 = cx + half_w, cy + half_h

            # -------------------------------------------------
            # Traffic light behaviour
            # -------------------------------------------------
            if color == "Red":
                got.mecanum_stop()
                action = "STOP"

            elif color == "Yellow":
                got.mecanum_move_speed(0, YELLOW_SPEED)
                action = "SLOW"

            elif color == "Green":
                got.mecanum_move_speed(0, GREEN_SPEED)
                action = "MOVE"

            elif color == "Blue":
                got.mecanum_turn_speed(2, 45)
                action = "TURN"

            else:
                got.mecanum_stop()
                action = "Unknown colour - STOP"

            # Draw bounding box and center dot
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.circle(img, (cx, cy), 4, (0, 0, 255), -1)

            # Label with color, shape, and area
            lines = [
                f"{color} {shape} width: {w}, area: {area}",
                f"cx: {cx} cy: {cy}",
                f"Action: {action}"
            ]

            line_height = 25  # adjust based on font scale
            for i, line in enumerate(lines):
                y = (y1 - 10) - (len(lines) - 1 - i) * line_height
                cv2.putText(img, line, (x1, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Display the frame
        cv2.imshow("Colour Recognition", img)

        # Press 'q' to quit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            print("Closing camera window...")
            break

finally:
    got.mecanum_stop()

    cv2.destroyAllWindows()
    cv2.waitKey(1)

Closing camera window...
